# Simple RAG（第一部分：本地运行准备）

本节目标：完成依赖导入、读取 PDF、完成切分（chunking），确保后续能顺利进入“向量库 + 检索”的核心流程。


## 0) 导入依赖并加载环境变量

下面只做两件事：

- 加载 `.env`（用于读取 `OPENAI_API_KEY`、`OPENAI_BASE_URL`）
- 导入本节会用到的包


In [1]:
from __future__ import annotations

import os
from pathlib import Path

from dotenv import load_dotenv

load_dotenv("../.env")

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

from langchain_openai import OpenAIEmbeddings

/tmp/ipykernel_1938448/3032572773.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


## 1) 准备 PDF

如需换成你自己的 PDF，只需要改 `path`。


In [2]:
path = Path("../data/Understanding_Climate_Change.pdf")
assert path.exists(), f"找不到 PDF: {path.resolve()}"

path

PosixPath('../data/Understanding_Climate_Change.pdf')

## 2) 读取 PDF 并切分

这一节我们只验证两步：

- 能成功读取 PDF
- 能按固定参数切分为 chunks


In [3]:
loader = PyPDFLoader(str(path))
docs = loader.load()

len(docs), docs[0].page_content[:200]

(33,
 'Understanding Climate Change \nChapter 1: Introduction to Climate Change \nClimate change refers to significant, long-term changes in the global climate. The term \n"global climate" encompasses the plane')

In [4]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

len(splits), splits[0].page_content[:200]

(97,
 'Understanding Climate Change \nChapter 1: Introduction to Climate Change \nClimate change refers to significant, long-term changes in the global climate. The term \n"global climate" encompasses the plane')

## 3) 构建向量库并创建 retriever

把 chunks 编码成向量并写入 FAISS，然后得到一个 retriever。

In [5]:
embeddings = OpenAIEmbeddings()
chunks_vector_store = FAISS.from_documents(splits, embeddings)

chunks_query_retriever = chunks_vector_store.as_retriever(search_kwargs={"k": 2})
chunks_query_retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x79560ce6eb40>, search_kwargs={'k': 2})

## 4) 测试检索

先不生成回答，只检查检索到的 chunks 是否相关。

In [6]:
test_query = "What is the main cause of climate change?"
retrieved_docs = chunks_query_retriever.invoke(test_query)

len(retrieved_docs), retrieved_docs[0].page_content[:300]

(2,
 'Chapter 2: Causes of Climate Change \nGreenhouse Gases \nThe primary cause of recent climate change is the increase in greenhouse gases in the \natmosphere. Greenhouse gases, such as carbon dioxide (CO2), methane (CH4), and nitrous \noxide (N2O), trap heat from the sun, creating a "greenhouse effect." T')

In [7]:
for i, d in enumerate(retrieved_docs, start=1):
    print(f"\n--- chunk {i} ---")
    print(d.page_content[:800])


--- chunk 1 ---
Chapter 2: Causes of Climate Change 
Greenhouse Gases 
The primary cause of recent climate change is the increase in greenhouse gases in the 
atmosphere. Greenhouse gases, such as carbon dioxide (CO2), methane (CH4), and nitrous 
oxide (N2O), trap heat from the sun, creating a "greenhouse effect." This effect is essential 
for life on Earth, as it keeps the planet warm enough to support life. However, human 
activities have intensified this natural process, leading to a warmer climate. 
Fossil Fuels 
Burning fossil fuels for energy releases large amounts of CO2. This includes coal, oil, and 
natural gas used for electricity, heating, and transportation. The industrial revolution marked 
the beginning of a significant increase in fossil fuel consumption, which continues to rise 
today. 
Coa

--- chunk 2 ---
Most of these climate changes are attributed to very small variations in Earth's orbit that 
change the amount of solar energy our planet receives. During the Holoce

## 5) 基于检索结果生成回答（可选）

用检索到的上下文作为输入，让模型生成答案。

In [8]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

context = "\n\n".join(d.page_content for d in retrieved_docs)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You answer questions using only the provided context. If the answer is not in the context, say you don't know."),
        ("human", "Context:\n{context}\n\nQuestion: {question}"),
    ]
)

chain = prompt | llm

answer = chain.invoke({"context": context, "question": test_query})
answer.content

'The main cause of recent climate change is the increase in greenhouse gases in the atmosphere.\n\nThese gases—especially carbon dioxide (CO₂), methane (CH₄), and nitrous oxide (N₂O)—trap heat from the sun and intensify the natural greenhouse effect, leading to a warmer climate. The context also specifies that human activities, particularly the burning of fossil fuels, are responsible for the increased emissions of these gases.'

## 6) 展示检索到的上下文（对齐原版）

使用仓库自带的 `helper_functions` 展示检索结果。

In [ ]:
import sys
from pathlib import Path

# 使用本 CN 目录下的 helper_functions
cn_root = (Path.cwd() / "..").resolve()  # RAG_Techniques_CN
sys.path.append(str(cn_root))

from helper_functions import retrieve_context_per_question, show_context

context_list = retrieve_context_per_question(test_query, chunks_query_retriever)
show_context(context_list)

Context 1:
Chapter 2: Causes of Climate Change 
Greenhouse Gases 
The primary cause of recent climate change is the increase in greenhouse gases in the 
atmosphere. Greenhouse gases, such as carbon dioxide (CO2), methane (CH4), and nitrous 
oxide (N2O), trap heat from the sun, creating a "greenhouse effect." This effect is essential 
for life on Earth, as it keeps the planet warm enough to support life. However, human 
activities have intensified this natural process, leading to a warmer climate. 
Fossil Fuels 
Burning fossil fuels for energy releases large amounts of CO2. This includes coal, oil, and 
natural gas used for electricity, heating, and transportation. The industrial revolution marked 
the beginning of a significant increase in fossil fuel consumption, which continues to rise 
today. 
Coal

Context 2:
Most of these climate changes are attributed to very small variations in Earth's orbit that 
change the amount of solar energy our planet receives. During the Holocene epoch, 

## 7) 评估（对齐原版）

调用仓库自带的 `evaluate_rag` 对 retriever 做一次快速评估。

In [10]:
import sys
from pathlib import Path

# 使用本 CN 目录下的 evaluation
cn_root = (Path.cwd() / "..").resolve()  # references/RAG_Techniques_CN
sys.path.append(str(cn_root))

from evaluation.evalute_rag import evaluate_rag

eval_result = evaluate_rag(chunks_query_retriever)
eval_result

{'questions': ['What are the main human activities contributing to climate change today?',
  'How does increased atmospheric CO₂ affect global average temperatures over time?',
  'What are the potential impacts of rising sea levels on coastal cities and ecosystems?',
  'How do feedback loops, such as melting ice reducing albedo, accelerate climate change?',
  'What policy measures are most effective in reducing greenhouse gas emissions globally?'],
 'results': ['```json\n{\n  "Relevance": 4,\n  "Completeness": 3,\n  "Conciseness": 2\n}\n```',
  '```json\n{\n  "Relevance": 3,\n  "Completeness": 1,\n  "Conciseness": 2\n}\n```',
  '```json\n{\n  "Relevance": 4,\n  "Completeness": 3,\n  "Conciseness": 2\n}\n```',
  '{\n  "Relevance": 3,\n  "Completeness": 2,\n  "Conciseness": 3\n}',
  '```json\n{\n  "Relevance": 5,\n  "Completeness": 4,\n  "Conciseness": 3\n}\n```'],
 'average_scores': {'Relevance': 3.8, 'Completeness': 2.6, 'Conciseness': 2.4}}